### Complete HT table results  
We would like to include **p-values, overlap rate, intensity and confidence scores**.   
Allow easy adding cell types and qualified sections.

In [4]:
source("./scale.R")
test <- read.csv("/data/coro_data.csv")
test = test %>% rename(x = x_section, y = y_section)
options(warn=-1)

Loading required package: spatstat.data

Loading required package: spatstat.univar

spatstat.univar 3.1-1

Loading required package: spatstat.geom

spatstat.geom 3.3-4

Loading required package: spatstat.random

spatstat.random 3.3-2

Loading required package: spatstat.explore

Loading required package: nlme

spatstat.explore 3.3-3

Loading required package: spatstat.model

Loading required package: rpart

spatstat.model 3.3-3

Loading required package: spatstat.linnet

spatstat.linnet 3.2-3


spatstat 3.3-0 
For an introduction to spatstat, type ‘beginner’ 



Attaching package: ‘dplyr’


The following object is masked from ‘package:nlme’:

    collapse


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
seeds <- sample(1:100,5,replace=FALSE)
seeds

[1] 25 42 69 17 61

In [3]:
library('foreach')
library('doParallel')
cores=detectCores()
cl <- makeCluster(cores[1]-1) #not to overload your computer
registerDoParallel(cl)

Loading required package: iterators

Loading required package: parallel



### HT table calculation

In [4]:
ecdf_fun <- function(x,perc) ecdf(x)(perc)
## Calculate column stats for each type
calc_pp_2d<-function(typ,pp,ps,section){
    nnp <- c()
    nni <- c()
    mnn <- c()
    res<- list()
    for (i in 1:length(pp)){     
        perc<- ecdf_fun(nndist(pp[[i]]), 0.01)
        int<- intensity(pp[[i]])
        mval<- mean(marks(pp[[i]]))
        nnp<-append(nnp,perc)
        nni<-append(nni,int)
        mnn <- append(mnn, mval)
        res[[as.character(section[i])]] = min(ps[,i])    
    }   
    pval <- signif(fisher(unname(unlist(res)))$p,2)
    res[['pval']] = pval
    res[['perc']] = round(mean(nnp)*100,2)
    res[['lam']] = round(mean(nni))
    res[['conf']] = round(mean(mnn),2)
    res[['rm']] = length(ps[,1])
    res[['Cluster']] = typ
    return(res)
}

### Excitatory clusters
We get the list of types and sections from the filtered table.

In [5]:
df<- read.csv("filter-exc.csv")
df1 <- replace(df, is.na(df), 0)
df1[df1<=0.5] <- NA
df1<-df1[rowSums(is.na(df1)) != ncol(df1)-1, ]
df1<-df1[!df1$cluster%in%(c('0109 L2/3 IT CTX Glut_2')), ]
row.names(df1) <- NULL

In [6]:
types = df1$cluster
types

[1] "0042 L6 IT CTX Glut_2"   "0061 L5 IT CTX Glut_3"  
 [3] "0070 L4/5 IT CTX Glut_1" "0072 L4/5 IT CTX Glut_1"
 [5] "0077 L4/5 IT CTX Glut_2" "0078 L4/5 IT CTX Glut_2"
 [7] "0082 L4/5 IT CTX Glut_2" "0084 L4/5 IT CTX Glut_3"
 [9] "0087 L4/5 IT CTX Glut_3" "0088 L4/5 IT CTX Glut_3"
[11] "0091 L4/5 IT CTX Glut_4" "0094 L4/5 IT CTX Glut_4"
[13] "0095 L4/5 IT CTX Glut_5" "0098 L4/5 IT CTX Glut_5"
[15] "0104 L2/3 IT CTX Glut_1" "0105 L2/3 IT CTX Glut_1"
[17] "0117 L2/3 IT CTX Glut_4" "0118 L2/3 IT CTX Glut_4"
[19] "0350 L5 ET CTX Glut_1"   "0351 L5 ET CTX Glut_1"  
[21] "0364 L5 ET CTX Glut_2"   "0436 L6 CT CTX Glut_1"  
[23] "0444 L6 CT CTX Glut_2"   "0468 L5 NP CTX Glut_3"  
[25] "0473 L5 NP CTX Glut_4"

In [7]:
sections<- apply(df1[,-1], 1, function(i) colnames(df1[,-1])[ !is.na(i) ]) ## qualified colns
sections<- sapply(sections, function(x){as.numeric(gsub("X.", "", x))})

In [8]:
df = data.frame(matrix(vector(), 0, 9,
                dimnames=list(c(), c('Cluster','59','60','61',"pval", "perc", "lam",'conf','rm'))),
                stringsAsFactors=F)
df$Cluster <- as.character(df$Cluster)
df

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>


### Table results
We use the parallel version to implement the HT table for all excitatory clusters passing through the intensity filter.  

In [8]:
set.seed(seeds[1])
res1 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ps<-get_pp_2d(pp, rm, TRUE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])
        q
    }
}
res1

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.010,0.020,NA,1.9e-03,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.045,NA,NA,4.5e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.005,0.005,0.020,6.0e-05,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.010,0.005,3.3e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [ ]:
write.csv(res1, "table-ex1.csv",row.names = FALSE)

In [9]:
set.seed(seeds[2])
res2 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ps<-get_pp_2d(pp, rm, TRUE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])
        q
    }
}
res2

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.005,0.015,NA,7.9e-04,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.040,NA,NA,4.0e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.010,0.005,0.055,2.6e-04,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.010,0.005,3.3e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [ ]:
write.csv(res2, "table-ex2.csv",row.names = FALSE)

In [10]:
set.seed(seeds[3])
res3 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ps<-get_pp_2d(pp, rm, TRUE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])
        q
    }
}
res3

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.005,0.020,NA,1.0e-03,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.020,NA,NA,2.0e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.010,0.005,0.025,1.3e-04,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.015,0.005,4.7e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [ ]:
write.csv(res3, "table-ex3.csv",row.names = FALSE)

In [11]:
set.seed(seeds[4])
res4 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ps<-get_pp_2d(pp, rm, TRUE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])
        q
    }
}
res4

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.005,0.010,NA,5.5e-04,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.030,NA,NA,3.0e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.005,0.005,0.015,4.7e-05,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.015,0.005,4.7e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [ ]:
write.csv(res4, "table-ex4.csv",row.names = FALSE)

In [12]:
set.seed(seeds[5])
res5 <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ps<-get_pp_2d(pp, rm, TRUE)
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,ps,sections[[t]])
        q
    }
}
res5

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.005,0.015,NA,7.9e-04,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.040,NA,NA,4.0e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.005,0.005,0.050,1.3e-04,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.010,0.005,3.3e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [ ]:
write.csv(res5, "table-ex5.csv",row.names = FALSE)

In [1]:
res1<-read.csv("./table-ex1.csv")
res2<-read.csv("./table-ex2.csv")
res3<-read.csv("./table-ex3.csv")
res4<-read.csv("./table-ex4.csv")
res5<-read.csv("./table-ex5.csv")

In [5]:
pv = c()
for(i in 1:nrow(res)) {
    row <- na.omit(unlist(unname(res[i,2:4])))
    pval <- signif(fisher(row)$p,2)
    pv<-append(pv,pval)
}
pv

[1] 9.8e-04 3.5e-02 5.0e-03 1.8e-05 2.9e-04 1.2e-04 3.9e-05 1.8e-05 2.9e-04
[10] 2.5e-01 1.8e-05 7.0e-03 1.8e-05 6.0e-04 2.5e-05 1.8e-05 2.7e-05 1.0e-02
[19] 2.0e-02 4.0e-05 4.8e-02 2.9e-04 5.0e-03 1.3e-01 3.0e-01

In [6]:
res<-Reduce("+", list(res1[,2:6],res2[,2:6],res3[,2:6],res4[,2:6],res5[,2:6])) / 5
res<-data.frame(Cluster=res1$Cluster, res)
res$pval<-pv
res$perc<-res1$perc
res$lam<-res1$lam
res$conf<-res1$conf
res$rm<-res1$rm
res

Cluster,X59,X60,X61,pval,perc,lam,conf,rm
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>
0042 L6 IT CTX Glut_2,0.006,0.016,NA,9.8e-04,0.98,318,0.70,20
0061 L5 IT CTX Glut_3,0.035,NA,NA,3.5e-02,3.96,268,0.68,23
0070 L4/5 IT CTX Glut_1,0.005,NA,NA,5.0e-03,2.20,302,0.70,34
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21
0077 L4/5 IT CTX Glut_2,NA,0.005,0.005,2.9e-04,1.39,427,0.70,23
0078 L4/5 IT CTX Glut_2,0.007,0.005,0.033,1.2e-04,1.42,279,0.73,24
0082 L4/5 IT CTX Glut_2,0.005,0.012,0.005,3.9e-05,0.00,423,0.71,29
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29
0087 L4/5 IT CTX Glut_3,NA,0.005,0.005,2.9e-04,1.32,317,0.72,24


In [7]:
write.csv(res, "table-ex.csv",row.names = FALSE)